# Single-Model Dual-Pass RLHF (PPO): Eliminating the Reference Model Copy

This notebook provides a complete guide and benchmark comparing standard **PPO RLHF** (which loads separate Actor and Reference models) against **Single-Model Dual-Pass PPO**.

By dynamically deactivating and re-activating PEFT (e.g. LoRA) adapters on a single copy of model weights during training, this strategy **eliminates the frozen Reference model footprint entirely**, saving up to 50% of the policy network VRAM. This makes alignment sweeps (RLHF/PPO) possible on consumer hardware (8GB - 16GB VRAM).

## 1. Install Dependencies

In [ ]:
# Install transformers, peft, accelerate, and dataset dependencies
!pip install -q transformers datasets peft accelerate tqdm

## 2. Core Implementation: Single-Model PPO Loss and Dual-Pass Step
Here is the clean, production-ready implementation of `SingleModelPPOLoss` and the dual-pass PPO training helper.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import PeftModel

class SingleModelPPOLoss(nn.Module):
    def __init__(self, kl_beta=0.01, clip_range=0.2, value_clip_range=0.2):
        super().__init__()
        self.kl_beta = kl_beta
        self.clip_range = clip_range
        self.value_clip_range = value_clip_range

    def get_log_probs(self, logits, labels):
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        log_probs = F.log_softmax(shift_logits, dim=-1)
        
        mask = (shift_labels != -100)
        gather_labels = shift_labels.clone()
        gather_labels[~mask] = 0
        selected_log_probs = torch.gather(log_probs, dim=-1, index=gather_labels.unsqueeze(-1)).squeeze(-1)
        selected_log_probs = selected_log_probs * mask.float()
        return selected_log_probs, mask

    def forward(self, model, inputs, action_masks, old_log_probs, advantages, returns, old_values=None, value_head=None):
        is_peft = isinstance(model, PeftModel)
        input_ids = inputs["input_ids"]
        attention_mask = inputs.get("attention_mask")
        
        # 1. REFERENCE PASS: Disable adapters and run under torch.no_grad()
        if is_peft:
            with model.disable_adapter():
                with torch.no_grad():
                    ref_outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        use_cache=False,
                        output_hidden_states=True
                    )
                    ref_logits = ref_outputs.logits.detach()
        else:
            with torch.no_grad():
                ref_outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                    output_hidden_states=True
                )
                ref_logits = ref_outputs.logits.detach()
                
        ref_log_probs, token_mask = self.get_log_probs(ref_logits, input_ids)
        ref_log_probs = ref_log_probs * action_masks[..., 1:]
        
        # 2. ACTOR PASS: Run with active adapters
        actor_outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=True
        )
        actor_logits = actor_outputs.logits
        actor_log_probs, _ = self.get_log_probs(actor_logits, input_ids)
        actor_log_probs = actor_log_probs * action_masks[..., 1:]
        
        # 3. PPO POLICY LOSS
        ratios = torch.exp(actor_log_probs - old_log_probs[..., 1:])
        surr1 = ratios * advantages[..., 1:]
        surr2 = torch.clamp(ratios, 1.0 - self.clip_range, 1.0 + self.clip_range) * advantages[..., 1:]
        
        mask_float = (token_mask * action_masks[..., 1:]).float()
        policy_loss = -torch.sum(torch.min(surr1, surr2) * mask_float, dim=-1)
        policy_loss = torch.mean(policy_loss / (torch.sum(mask_float, dim=-1) + 1e-8))
        
        # 4. KL PENALTY
        kl_div = actor_log_probs - ref_log_probs
        kl_penalty = torch.sum(kl_div * mask_float, dim=-1)
        kl_penalty = torch.mean(kl_penalty / (torch.sum(mask_float, dim=-1) + 1e-8))
        
        total_loss = policy_loss + self.kl_beta * kl_penalty
        
        # 5. VALUE LOSS (Critic)
        value_loss = torch.tensor(0.0, device=input_ids.device)
        if value_head is not None and old_values is not None:
            hidden_states = actor_outputs.hidden_states[-1]
            values = value_head(hidden_states).squeeze(-1)
            values = values[..., :-1] * action_masks[..., 1:]
            
            values_clipped = old_values[..., 1:] + torch.clamp(
                values - old_values[..., 1:], -self.value_clip_range, self.value_clip_range
            )
            vf_losses1 = (values - returns[..., 1:]) ** 2
            vf_losses2 = (values_clipped - returns[..., 1:]) ** 2
            vf_loss = 0.5 * torch.sum(torch.max(vf_losses1, vf_losses2) * mask_float, dim=-1)
            value_loss = torch.mean(vf_loss / (torch.sum(mask_float, dim=-1) + 1e-8))
            total_loss = total_loss + value_loss
            
        return total_loss, policy_loss.item(), kl_penalty.item(), value_loss.item()

def step_ppo_dual_pass(model, inputs, action_masks, old_log_probs, advantages, returns, optimizer, 
                       old_values=None, value_head=None, kl_beta=0.01, clip_range=0.2):
    model.train()
    if value_head is not None:
        value_head.train()
    optimizer.zero_grad()
    
    loss_fn = SingleModelPPOLoss(kl_beta=kl_beta, clip_range=clip_range)
    total_loss, p_loss, kl_pen, val_loss = loss_fn(
        model=model,
        inputs=inputs,
        action_masks=action_masks,
        old_log_probs=old_log_probs,
        advantages=advantages,
        returns=returns,
        old_values=old_values,
        value_head=value_head
    )
    
    total_loss.backward()
    optimizer.step()
    return total_loss.item(), p_loss, kl_pen, val_loss

## 3. Run Memory Benchmark
Let's measure the VRAM difference between standard PPO (separate Actor and Reference models) and Single-Model PPO on your GPU.

In [ ]:
# Run the PPO benchmark script comparing VRAM peak footprint
!python benchmark_rlhf.py --model_name Qwen/Qwen2.5-3B-Instruct

## 4. How to integrate into custom PPO Training Loops
You can easily integrate this class into custom RLHF libraries (like Hugging Face TRL or custom training loops) by replacing standard loss evaluations with the dual-pass logic:

In [ ]:
# Boilerplate guide:
# from rlhf_dual_pass import step_ppo_dual_pass
#
# for epoch in range(num_epochs):
#     for batch in dataloader:
#         loss, p_loss, kl, v_loss = step_ppo_dual_pass(
#             model=actor_lora_model,
#             inputs=batch_inputs,
#             action_masks=batch_action_masks,
#             old_log_probs=rollout_log_probs,
#             advantages=computed_advantages,
#             returns=computed_returns,
#             optimizer=optimizer,
#             old_values=rollout_values,
#             value_head=critic_head_layer
#         )